# KA SS 26 · Challenge 2 — **Stage 9: cross-family blend (SSL × non-SSL)**

> This notebook was Stage 7. Cells 1–13 are unchanged and now serve as the **loader** for the
> Stage 7 artifacts in `/kaggle/working`. Cells 14–18 are the old Stage 7 sweep and are
> **skipped**. Cells S1–S5 at the bottom are Stage 9.

### What changed since Stage 7

Stage 7 scored **0.5259** public against Stage 5's **0.5293**, and the calibration line predicted
0.510 for its local 0.8035 — a residual of **+0.016**, half of Stage 6's +0.033. Inside the SSL
family, **+0.0295 local bought +0.005 public** (a transfer ratio near 0.17, not 0.77). The SSL
bonus shrinks as local F1 rises; it is not a constant to add.

Stage 7 also showed the ensemble was too homogeneous: three siblings, members at 0.7999 / 0.7938 /
0.7866, ensemble **0.8035** — **+0.0036** over the best member for two extra 70-epoch runs.

And `t_seed = 0.65` won *again* with 0.90 above it in the grid. It is a confirmed interior
optimum. The seed **threshold** is not the binding constraint; the seed **target** is.

### What Stage 9 does

Stage 8 (Colab) trained the other family — **plain ImageNet-21k ConvNeXt-UNet, no SimSiam** — as
two members: `n1` at the Stage 5 control erosion 0.45, `n2` at 0.58 to attack under-segmentation
(test predictions still run ~108/image against a GT mean of 125.2).

Stage 9 blends the two families' probability maps and sweeps the blend weight:

blend = w · mean(n1, n2) + (1 − w) · mean(m1, m2, m3)

Both families were scored on the **same 86 validation images**. S1 verifies that against
`val_ids.json` rather than trusting a count — Stage 7 was configured to reproduce Stage 5's split
and silently produced a different one, which is exactly the failure this check catches.

S3 then tries **density-adaptive thresholds**: one global decoder over images holding 15 cells and
images holding 600 is crude, so the images are binned by predicted foreground fraction (computable
on test) and each bin gets its own `(t_cell, t_seed, min_area)` — accepted only at **+0.005**,
because 86 images over three bins is ~29 each and that is thin.

### Reading the numbers

Public differences under ~0.008 are inside the noise of a 91-image split. Stage 5's 0.5293 beating
Stage 7's 0.5259 by 0.0034 is **under one standard error** — it is not evidence Stage 5 is better.
Keep `5th_submission.csv` selected as the anchor; the second slot goes to the blend, which has the
better expected *private* score than either family alone even if public says otherwise.

In [1]:
# ── 1 · Imports and environment ────────────────────────────────────────────────
import os, sys, gc, json, math, time, random, shutil, pickle, subprocess, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import scipy.ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_dilation
from skimage.segmentation import watershed
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

try:
    import cv2
    HAVE_CV2 = True
except ImportError:
    HAVE_CV2 = False

# AMP works across torch 1.x and 2.x
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

def amp_autocast():
    return _autocast("cuda", dtype=torch.float16) if _AMP_NEW else _autocast()

def make_scaler():
    return _GradScaler("cuda") if _AMP_NEW else _GradScaler()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

T_START = time.time()
def elapsed():
    return time.time() - T_START

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}h {(sec % 3600) // 60:02d}m {sec % 60:02d}s"

print("torch      :", torch.__version__)
print("timm       :", timm.__version__)
print("device     :", DEVICE)
print("opencv     :", HAVE_CV2)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  gpu {i}    : {p.name} ({p.total_memory / 1e9:.1f} GB)")
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 x2 in Session options."

torch      : 2.10.0+cu128
timm       : 1.0.26
device     : cuda
opencv     : True
  gpu 0    : Tesla T4 (15.6 GB)
  gpu 1    : Tesla T4 (15.6 GB)


In [2]:
# ── 2 · Configuration ──────────────────────────────────────────────────────────
class CFG:
    RUN_TAG = "stage7_ssl_ensemble"

    # Set True for a ~12 minute end-to-end rehearsal of every cell (1 member, 3 epochs,
    # tiny grids). Use it once if you want to validate the pipeline before the real run.
    SMOKE_TEST = False

    RESUME = True          # skip phases whose artifacts already exist
    RESET_WORKDIR = False  # DANGER: wipes ckpt/ and artifacts/. Never True while recovering.

    # ── wall-clock budgets ────────────────────────────────────────────────────
    TOTAL_BUDGET_SEC   = 9.0 * 3600
    SSL_BUDGET_SEC     = 40 * 60
    MEMBER_BUDGET_SEC  = 80 * 60
    FINAL_RESERVE_SEC  = 60 * 60   # phases D-F must fit in this

    # ── geometry (re-detected from disk) ──────────────────────────────────────
    IMG_H, IMG_W = 520, 704
    PAD_MULT = 32

    # ── split: identical policy to Stage 5 so local F1 stays comparable ───────
    SPLIT_SEED = 2026
    GROUP_SPLIT = True
    DUP_CORR_THR = 0.72
    N_FOLDS = 5
    FOLD = 0
    CLUSTER_K = 2

    # ── architecture ──────────────────────────────────────────────────────────
    ENCODER = "convnext_tiny.fb_in22k_ft_in1k"
    ENCODER_FALLBACKS = ("convnext_tiny", "resnet50", "resnet34")
    DECODER_CHANNELS = (256, 128, 64, 32)
    OUT_CHANNELS = 2  # 0 = foreground union, 1 = eroded instance seed

    # ── targets (Stage 5 values) ──────────────────────────────────────────────
    SEED_EROSION_FRAC = 0.45
    BORDER_DILATION = 2
    BORDER_WEIGHT = 4.0

    # ── phase A: SimSiam domain adaptation ────────────────────────────────────
    RUN_SSL = True
    SSL_EPOCHS = 25          # Stage 6 used 8 -> the one axis with proven transfer, barely touched
    SSL_SIZE = 288           # Stage 6 used 224
    SSL_BATCH = 24
    SSL_LR = 1.5e-4
    SSL_WD = 1e-4
    SSL_PROJ_DIM = 256
    SSL_HIDDEN_DIM = 768
    SSL_INCLUDE_TRAIN_IMAGES = True

    # ── phase B: supervised members (Stage 5 schedule) ────────────────────────
    SUP_EPOCHS = 70
    SUP_BATCH = 8
    SUP_LR = 3e-4
    SUP_ENCODER_LR_MULT = 0.25
    SUP_WD = 1e-4
    FREEZE_ENCODER_EPOCHS = 2
    WARMUP_STEPS = 150
    STEPS_PER_EPOCH = 42
    EVAL_START = 24
    EVAL_EVERY = 2
    GRAD_CLIP = 5.0

    # All members share fold 0 so map averaging and threshold tuning stay honest.
    MEMBERS = [
        dict(name="m1", seed=2026, crop=(448, 576)),
        dict(name="m2", seed=43,   crop=(400, 512)),
        dict(name="m3", seed=7,    crop=(480, 608)),
    ]

    # ── augmentation ──────────────────────────────────────────────────────────
    AUG_HFLIP = 0.5
    AUG_VFLIP = 0.5
    AUG_SCALE = (0.85, 1.20)
    AUG_BRIGHT = 0.15
    AUG_CONTRAST = 0.15
    AUG_NOISE = 0.02

    # ── inference ─────────────────────────────────────────────────────────────
    TTA_FLIPS = True
    TTA_SCALES = (0.85, 1.0, 1.15)

    # ── phase D: decode sweep ─────────────────────────────────────────────────
    # t_seed hit the top of the grid in BOTH Stage 5 (0.65) and Stage 6 (0.65) while
    # predictions/image ran ~100 against a GT mean of 125 -> push the ceiling up.
    T_CELL_GRID = [0.45, 0.50, 0.55, 0.60, 0.65]
    T_SEED_GRID = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
    T_SEED_EXTENSION = [0.92, 0.94, 0.96]   # used only if the optimum lands on the edge
    MIN_AREA_GRID = [30, 50, 70, 90, 110, 130]   # GT area p1 is 53; Stage 5 used 87
    SCORE_THR_GRID = [0.0, 0.15, 0.25, 0.35, 0.45]
    SWEEP_SUBSET = 45        # coarse pass image count
    SWEEP_REFINE_TOP = 8     # combos re-scored on the full validation set
    PER_CLUSTER_MIN_GAIN = 0.005

    # fallback decode used for per-member submissions written before the sweep runs
    DEFAULT_POST = dict(t_cell=0.50, t_seed=0.70, min_area=60, score_thr=0.0)

    SUB_INDEX_COL = "idx"


if CFG.SMOKE_TEST:
    CFG.SSL_EPOCHS = 1
    CFG.SUP_EPOCHS = 3
    CFG.EVAL_START = 1
    CFG.EVAL_EVERY = 1
    CFG.MEMBERS = CFG.MEMBERS[:1]
    CFG.TTA_SCALES = (1.0,)
    CFG.T_CELL_GRID = [0.50, 0.55]
    CFG.T_SEED_GRID = [0.65, 0.75]
    CFG.MIN_AREA_GRID = [50, 90]
    CFG.SCORE_THR_GRID = [0.0]
    CFG.SWEEP_SUBSET = 12
    CFG.SSL_BUDGET_SEC = 6 * 60
    CFG.MEMBER_BUDGET_SEC = 10 * 60
    CFG.FINAL_RESERVE_SEC = 15 * 60
    print(">>> SMOKE TEST MODE - results are meaningless, this only proves the pipeline runs")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SPLIT_SEED)
STATS = {"run_tag": CFG.RUN_TAG, "smoke_test": CFG.SMOKE_TEST}
print("run tag:", CFG.RUN_TAG, "| members:", [m["name"] for m in CFG.MEMBERS])

run tag: stage7_ssl_ensemble | members: ['m1', 'm2', 'm3']


In [3]:
# ── 3 · Paths, working directory, preflight checks ─────────────────────────────
WORK = Path("/kaggle/working")
CKPT = WORK / "ckpt"
ART  = WORK / "artifacts"
CACHE = WORK / "cache"
MAPS = WORK / "maps"

if CFG.RESET_WORKDIR:
    print("!! RESET_WORKDIR is True - clearing ckpt/, artifacts/, maps/")
    for p in (CKPT, ART, MAPS):
        if p.exists():
            shutil.rmtree(p)

for p in (CKPT, ART, CACHE, MAPS):
    p.mkdir(parents=True, exist_ok=True)


def find_dataset_root():
    base = Path("/kaggle/input")
    assert base.exists(), "No /kaggle/input. Attach the competition under Input -> Add Input."
    for csv_path in sorted(base.rglob("train.csv")):
        d = csv_path.parent
        if (d / "train").is_dir() and (d / "test").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find the competition data. Add Input -> Competitions -> KA SS 26: Challenge 2."
    )

BASE = find_dataset_root()
TRAIN_DIR, TEST_DIR, TRAIN_CSV = BASE / "train", BASE / "test", BASE / "train.csv"

UNLABELED_DIR = None
for cand in sorted(BASE.rglob("*")):
    if cand.is_dir() and "unlabeled" in cand.name.lower():
        UNLABELED_DIR = cand
        break
UNLABELED_PATHS = sorted(UNLABELED_DIR.glob("*.png")) if UNLABELED_DIR else []

SAMPLE_SUB = next(iter(BASE.glob("sample_submission.csv")), None)

print("data root :", BASE)
print("train     :", len(list(TRAIN_DIR.glob("*.png"))), "images")
print("test      :", len(list(TEST_DIR.glob("*.png"))), "images")
print("unlabeled :", len(UNLABELED_PATHS), "images", f"({UNLABELED_DIR})" if UNLABELED_DIR else "(NOT FOUND)")
print("sample sub:", SAMPLE_SUB)

if not UNLABELED_PATHS:
    print("!! No unlabeled directory found - phase A will fall back to train images only.")
    CFG.SSL_INCLUDE_TRAIN_IMAGES = True

# Preflight: force the pretrained-weight download NOW rather than at hour three.
def resolve_encoder():
    names = [CFG.ENCODER] + list(CFG.ENCODER_FALLBACKS)
    for name in names:
        try:
            m = timm.create_model(name, pretrained=True, features_only=True, in_chans=3)
            ch = m.feature_info.channels()
            del m; gc.collect()
            print(f"encoder ok: {name} | feature channels {ch}")
            return name
        except Exception as exc:
            print(f"  [{name}] unavailable: {type(exc).__name__}: {str(exc)[:160]}")
    raise RuntimeError(
        "No pretrained encoder could be downloaded.\n"
        "Turn Internet ON in Session options (Notebook -> Settings -> Internet), or attach a "
        "dataset containing timm weights. Without ImageNet-21k weights this notebook is pointless."
    )

CFG.ENCODER = resolve_encoder()
STATS["encoder"] = CFG.ENCODER

data root : /kaggle/input/competitions/ka-ss-26-challenge-2/cells_segmentation
train     : 424 images
test      : 182 images
unlabeled : 1972 images (/kaggle/input/competitions/ka-ss-26-challenge-2/cells_segmentation/unlabeled_additional_data)
sample sub: None


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

encoder ok: convnext_tiny.fb_in22k_ft_in1k | feature channels [96, 192, 384, 768]


In [4]:
# ── 4 · Labels, RLE codec, and a round-trip assertion ──────────────────────────
# The codec below is byte-identical to the one that produced the 0.5293 and 0.5209
# submissions. The assertion at the end is the cheap version of the scorer-parity
# check: if encode->decode is lossy, local F1 and public score diverge silently.

df_raw = pd.read_csv(TRAIN_CSV)
ID_COL = next(c for c in df_raw.columns if c.lower() in ("id", "image_id", "imageid"))
ANN_COL = next(c for c in df_raw.columns if c.lower() in ("annotation", "encodedpixels", "rle", "segmentation"))
print("columns:", df_raw.columns.tolist(), "| rows:", len(df_raw))

TRAIN_IDS = sorted(p.stem for p in TRAIN_DIR.glob("*.png"))
TEST_IDS = sorted(p.stem for p in TEST_DIR.glob("*.png"))
with Image.open(TRAIN_DIR / f"{TRAIN_IDS[0]}.png") as im:
    CFG.IMG_W, CFG.IMG_H = im.size
PAD_H = int(math.ceil(CFG.IMG_H / CFG.PAD_MULT) * CFG.PAD_MULT)
PAD_W = int(math.ceil(CFG.IMG_W / CFG.PAD_MULT) * CFG.PAD_MULT)
NPIX = CFG.IMG_H * CFG.IMG_W
print(f"image {CFG.IMG_H}x{CFG.IMG_W} -> padded {PAD_H}x{PAD_W} | {len(TRAIN_IDS)} train / {len(TEST_IDS)} test")


def rle_decode_flat(rle, h, w, order="C"):
    if not isinstance(rle, str) or not rle.strip() or rle.strip() == "-1":
        return np.empty(0, dtype=np.int64)
    values = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = values[0::2] - 1, values[1::2]
    idx = np.concatenate([np.arange(s, s + l) for s, l in zip(starts, lengths)])
    if order == "C":
        return idx
    return (idx % h) * w + (idx // h)


def rle_encode(mask_flat_u8):
    pixels = np.concatenate([[0], mask_flat_u8, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(int(x)) for x in runs)


def flat_to_rle(flat):
    mask = np.zeros(NPIX, dtype=np.uint8)
    mask[flat] = 1
    return rle_encode(mask)


# Detect the flattening order the organisers used (expected: C / row-major).
probe = df_raw[df_raw[ANN_COL].astype(str) != "-1"].sample(min(200, len(df_raw)), random_state=0)
order_report = {}
for order in ("C", "F"):
    single = []
    for rle in probe[ANN_COL].astype(str):
        flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, order)
        if flat.size == 0 or flat.max() >= NPIX:
            continue
        m = np.zeros(NPIX, bool); m[flat] = True
        single.append(ndi.label(m.reshape(CFG.IMG_H, CFG.IMG_W))[1] == 1)
    order_report[order] = float(np.mean(single) * 100) if single else 0.0
RLE_ORDER = max(order_report, key=lambda o: order_report[o])
print("single-component % by order:", {k: round(v, 1) for k, v in order_report.items()},
      "-> RLE_ORDER =", RLE_ORDER)
assert order_report[RLE_ORDER] > 90, "RLE order detection failed - inspect before continuing."

INST = defaultdict(list)
for image_id, rle in zip(df_raw[ID_COL].astype(str), df_raw[ANN_COL].astype(str)):
    flat = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
    if flat.size:
        INST[image_id].append(np.unique(flat))
counts = np.array([len(INST[i]) for i in TRAIN_IDS])
areas = np.array([len(f) for i in TRAIN_IDS for f in INST[i]])
print(f"instances: {int(counts.sum())} | per image p50 {int(np.percentile(counts, 50))} "
      f"p95 {int(np.percentile(counts, 95))} max {int(counts.max())} mean {counts.mean():.1f}")
print(f"instance area p1 {int(np.percentile(areas, 1))} p50 {int(np.percentile(areas, 50))} "
      f"p95 {int(np.percentile(areas, 95))}")
STATS["gt_mean_instances"] = float(counts.mean())

# ROUND-TRIP PARITY CHECK -------------------------------------------------------
bad = 0
for image_id in TRAIN_IDS[:40]:
    for flat in INST[image_id]:
        back = rle_decode_flat(flat_to_rle(flat), CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
        if not np.array_equal(np.unique(back), flat):
            bad += 1
assert bad == 0, f"RLE round-trip is lossy on {bad} instances - this alone would cost leaderboard points."
print(">>> RLE round-trip exact on 40 images. Encode/decode is not the source of the local-to-public gap.")

columns: ['Unnamed: 0', 'id', 'annotation'] | rows: 53065
image 520x704 -> padded 544x704 | 424 train / 182 test
single-component % by order: {'C': 91.5, 'F': 0.0} -> RLE_ORDER = C
instances: 53065 | per image p50 47 p95 503 max 782 mean 125.2
instance area p1 53 p50 215 p95 920
>>> RLE round-trip exact on 40 images. Encode/decode is not the source of the local-to-public gap.


In [5]:
# ── 5 · Images and training targets (cached) ───────────────────────────────────
def load_gray_u8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.uint8)

IMAGES = {i: load_gray_u8(TRAIN_DIR / f"{i}.png") for i in TRAIN_IDS}
print("loaded", len(IMAGES), "train images")

TARGET_CACHE = CACHE / "targets_v1.npz"


def build_targets_for(image_id):
    # channel 0 foreground union, 1 eroded seed, 2 contact/border zone
    h, w = CFG.IMG_H, CFG.IMG_W
    fg = np.zeros((h, w), np.uint8)
    seed = np.zeros((h, w), np.uint8)
    overlap = np.zeros((h, w), np.uint16)
    for flat in INST[image_id]:
        ys, xs = np.divmod(flat, w)
        y0, y1 = ys.min(), ys.max() + 1
        x0, x1 = xs.min(), xs.max() + 1
        sub = np.zeros((y1 - y0, x1 - x0), bool)
        sub[ys - y0, xs - x0] = True
        fg[y0:y1, x0:x1] |= sub
        dist = distance_transform_edt(np.pad(sub, 1))[1:-1, 1:-1]
        dmax = dist.max()
        if dmax > 0:
            core = dist > CFG.SEED_EROSION_FRAC * dmax
            if not core.any():
                core = dist >= dmax
            seed[y0:y1, x0:x1] |= core
        pad = CFG.BORDER_DILATION
        ys0, ys1 = max(0, y0 - pad), min(h, y1 + pad)
        xs0, xs1 = max(0, x0 - pad), min(w, x1 + pad)
        big = np.zeros((ys1 - ys0, xs1 - xs0), bool)
        big[y0 - ys0:y1 - ys0, x0 - xs0:x1 - xs0] = sub
        overlap[ys0:ys1, xs0:xs1] += binary_dilation(big, iterations=pad).astype(np.uint16)
    border = (overlap >= 2).astype(np.uint8)
    return np.stack([fg, seed, border], 0)


if CFG.RESUME and TARGET_CACHE.exists():
    with np.load(TARGET_CACHE) as z:
        TARGETS = {k: z[k] for k in z.files}
    print("targets loaded from cache:", TARGET_CACHE)
else:
    t0 = time.time()
    TARGETS = {}
    for n, image_id in enumerate(TRAIN_IDS, 1):
        TARGETS[image_id] = build_targets_for(image_id)
        if n % 100 == 0:
            print(f"  targets {n}/{len(TRAIN_IDS)} ({time.time() - t0:.0f}s)")
    np.savez_compressed(TARGET_CACHE, **TARGETS)
    print(f"targets built in {time.time() - t0:.0f}s -> {TARGET_CACHE}")

_t = TARGETS[TRAIN_IDS[0]]
print("target tensor:", _t.shape, _t.dtype,
      "| fg %", round(100 * _t[0].mean(), 1),
      "| seed %", round(100 * _t[1].mean(), 1),
      "| border %", round(100 * _t[2].mean(), 1))

MEAN = float(np.mean([IMAGES[i].mean() for i in TRAIN_IDS]) / 255.0)
STD = float(np.mean([IMAGES[i].std() for i in TRAIN_IDS]) / 255.0)
print(f"dataset mean/std = {MEAN:.4f}/{STD:.4f}")

loaded 424 train images
targets loaded from cache: /kaggle/working/cache/targets_v1.npz
target tensor: (3, 520, 704) uint8 | fg % 23.0 | seed % 7.7 | border % 3.6
dataset mean/std = 0.5018/0.0522


In [6]:
# ── 6 · Grouped split (fold 0) and the k=2 morphology proxy ────────────────────
TH_H, TH_W = 65, 88


def thumbnail_vec(arr):
    a = np.asarray(Image.fromarray(arr).resize((TH_W, TH_H), Image.BILINEAR), np.float32).ravel()
    a -= a.mean()
    n = np.linalg.norm(a)
    return a / n if n > 0 else a

THUMBS = np.stack([thumbnail_vec(IMAGES[i]) for i in TRAIN_IDS])
CORR = THUMBS @ THUMBS.T
np.fill_diagonal(CORR, 0.0)

if CFG.GROUP_SPLIT:
    adj = CORR >= CFG.DUP_CORR_THR
    group_of = -np.ones(len(TRAIN_IDS), int)
    g = 0
    for i in range(len(TRAIN_IDS)):
        if group_of[i] >= 0:
            continue
        stack, group_of[i] = [i], g
        while stack:
            u = stack.pop()
            for v in np.nonzero(adj[u])[0]:
                if group_of[v] < 0:
                    group_of[v] = g
                    stack.append(v)
        g += 1
    n_groups = g
else:
    group_of = np.arange(len(TRAIN_IDS))
    n_groups = len(TRAIN_IDS)

sizes = np.bincount(group_of)
print(f"groups: {n_groups} over {len(TRAIN_IDS)} images | size histogram "
      f"{dict(zip(*np.unique(sizes, return_counts=True)))}")

rng = np.random.RandomState(CFG.SPLIT_SEED)
order = rng.permutation(n_groups)
fold_of_group = {int(gid): int(k % CFG.N_FOLDS) for k, gid in enumerate(order)}
val_mask = np.array([fold_of_group[int(g)] == CFG.FOLD for g in group_of])
train_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if not m]
val_ids = [i for i, m in zip(TRAIN_IDS, val_mask) if m]

vi = np.nonzero(val_mask)[0]
ti = np.nonzero(~val_mask)[0]
max_corr = float(CORR[np.ix_(vi, ti)].max())
print(f"train/val images: {len(train_ids)}/{len(val_ids)} | max val-to-train correlation {max_corr:.3f} "
      f"(leak threshold {CFG.DUP_CORR_THR})")
assert max_corr < CFG.DUP_CORR_THR, "Near-duplicate leaked across the split."
STATS.update(n_groups=n_groups, max_val_train_corr=max_corr,
             n_train=len(train_ids), n_val=len(val_ids))

# Morphology proxy, fitted on train+test together so the assignment is available at test time.
ALL_IDS = TRAIN_IDS + TEST_IDS
ALL_THUMBS = np.concatenate([THUMBS, np.stack([
    thumbnail_vec(load_gray_u8(TEST_DIR / f"{i}.png")) for i in TEST_IDS])])
km = KMeans(n_clusters=CFG.CLUSTER_K, n_init=10, random_state=0).fit(ALL_THUMBS)
CLUSTER = {i: int(c) for i, c in zip(ALL_IDS, km.labels_)}
val_clusters = np.array([CLUSTER[i] for i in val_ids])
test_clusters = np.array([CLUSTER[i] for i in TEST_IDS])
print(f"cluster sizes  val {np.bincount(val_clusters, minlength=CFG.CLUSTER_K).tolist()} | "
      f"test {np.bincount(test_clusters, minlength=CFG.CLUSTER_K).tolist()}")

# Distribution-shift probe: does test look like train?
train_mix = np.bincount([CLUSTER[i] for i in TRAIN_IDS], minlength=CFG.CLUSTER_K) / len(TRAIN_IDS)
test_mix = np.bincount(test_clusters, minlength=CFG.CLUSTER_K) / len(TEST_IDS)
print(f"cluster mixture  train {np.round(train_mix, 3).tolist()} | test {np.round(test_mix, 3).tolist()}"
      f" | L1 shift {np.abs(train_mix - test_mix).sum():.3f}")
STATS["cluster_mix_train"] = train_mix.tolist()
STATS["cluster_mix_test"] = test_mix.tolist()

groups: 421 over 424 images | size histogram {np.int64(1): np.int64(418), np.int64(2): np.int64(3)}
train/val images: 338/86 | max val-to-train correlation 0.665 (leak threshold 0.72)
cluster sizes  val [31, 55] | test [85, 97]
cluster mixture  train [0.46, 0.54] | test [0.467, 0.533] | L1 shift 0.014


In [7]:
# ── 7 · Dataset and augmentation ───────────────────────────────────────────────
def resize_pair(img, tgt, scale):
    h, w = img.shape
    nh, nw = max(64, int(round(h * scale))), max(64, int(round(w * scale)))
    if HAVE_CV2:
        img_r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        tgt_r = np.stack([cv2.resize(tgt[c], (nw, nh), interpolation=cv2.INTER_NEAREST)
                          for c in range(tgt.shape[0])])
    else:
        img_r = np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize((nw, nh), Image.BILINEAR),
                           np.float32) / 255.0
        tgt_r = np.stack([np.asarray(Image.fromarray(tgt[c]).resize((nw, nh), Image.NEAREST))
                          for c in range(tgt.shape[0])])
    return img_r.astype(np.float32), tgt_r.astype(np.uint8)


def pad_to(arr, height, width, mode="reflect"):
    ph, pw = max(0, height - arr.shape[-2]), max(0, width - arr.shape[-1])
    if ph == 0 and pw == 0:
        return arr
    pad_width = [(0, 0)] * (arr.ndim - 2) + [(0, ph), (0, pw)]
    kw = {} if mode == "reflect" else {"constant_values": 0}
    return np.pad(arr, pad_width, mode=mode, **kw)


class CellDataset(Dataset):
    def __init__(self, ids, crop, training=True):
        self.ids = list(ids)
        self.crop_h, self.crop_w = crop
        self.training = bool(training)

    def __len__(self):
        return len(self.ids)

    def augment(self, img, tgt):
        if random.random() < CFG.AUG_HFLIP:
            img, tgt = img[:, ::-1], tgt[:, :, ::-1]
        if random.random() < CFG.AUG_VFLIP:
            img, tgt = img[::-1], tgt[:, ::-1]
        img = np.ascontiguousarray(img)
        tgt = np.ascontiguousarray(tgt)
        s = random.uniform(*CFG.AUG_SCALE)
        if abs(s - 1.0) > 1e-3:
            img, tgt = resize_pair(img, tgt, s)
        if CFG.AUG_BRIGHT > 0:
            img = img + random.uniform(-CFG.AUG_BRIGHT, CFG.AUG_BRIGHT) * STD
        if CFG.AUG_CONTRAST > 0:
            f = 1.0 + random.uniform(-CFG.AUG_CONTRAST, CFG.AUG_CONTRAST)
            img = (img - MEAN) * f + MEAN
        if CFG.AUG_NOISE > 0:
            img = img + np.random.randn(*img.shape).astype(np.float32) * CFG.AUG_NOISE * STD
        return np.clip(img, 0.0, 1.0).astype(np.float32), tgt

    def random_crop(self, img, tgt):
        img = pad_to(img, self.crop_h, self.crop_w)
        tgt = pad_to(tgt, self.crop_h, self.crop_w, mode="constant")
        h, w = img.shape
        y = random.randint(0, h - self.crop_h)
        x = random.randint(0, w - self.crop_w)
        return img[y:y + self.crop_h, x:x + self.crop_w], tgt[:, y:y + self.crop_h, x:x + self.crop_w]

    def __getitem__(self, index):
        image_id = self.ids[index]
        img = IMAGES[image_id].astype(np.float32) / 255.0
        tgt = TARGETS[image_id].copy()
        if self.training:
            img, tgt = self.augment(img, tgt)
            img, tgt = self.random_crop(img, tgt)
        else:
            img = pad_to(img, PAD_H, PAD_W)
            tgt = pad_to(tgt, PAD_H, PAD_W, mode="constant")
        weight = 1.0 + CFG.BORDER_WEIGHT * tgt[2].astype(np.float32)
        x = torch.from_numpy(np.ascontiguousarray(img)).unsqueeze(0).repeat(3, 1, 1)
        y = torch.from_numpy(np.ascontiguousarray(tgt[:2]).astype(np.float32))
        w = torch.from_numpy(np.ascontiguousarray(weight)).unsqueeze(0)
        return x, y, w, image_id


def normalize_batch(x):
    return (x - MEAN) / max(STD, 1e-6)


_ds = CellDataset(train_ids, CFG.MEMBERS[0]["crop"], training=True)
_x, _y, _w, _ = _ds[0]
print("sample batch shapes:", tuple(_x.shape), tuple(_y.shape), tuple(_w.shape),
      "| weight range", float(_w.min()), float(_w.max()))
del _ds, _x, _y, _w

sample batch shapes: (3, 448, 576) (2, 448, 576) (1, 448, 576) | weight range 1.0 5.0


In [8]:
# ── 8 · ConvNeXt-UNet ──────────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


def up_to(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


class UNet(nn.Module):
    def __init__(self, encoder_name, out_ch=2, dec=(256, 128, 64, 32), pretrained=True):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained,
                                         features_only=True, in_chans=3)
        chs = self.encoder.feature_info.channels()[-4:]
        self.stem = nn.Sequential(
            nn.Conv2d(3, dec[3], 3, padding=1, bias=False),
            nn.BatchNorm2d(dec[3]), nn.ReLU(inplace=True))
        self.d3 = ConvBlock(chs[3] + chs[2], dec[0])
        self.d2 = ConvBlock(dec[0] + chs[1], dec[1])
        self.d1 = ConvBlock(dec[1] + chs[0], dec[2])
        self.d0 = ConvBlock(dec[2], dec[3])
        self.dfull = ConvBlock(dec[3] * 2, dec[3])
        self.head = nn.Conv2d(dec[3], out_ch, 1)

    def forward(self, x):
        stem = self.stem(x)
        f1, f2, f3, f4 = self.encoder(x)[-4:]
        d = self.d3(torch.cat([up_to(f4, f3), f3], 1))
        d = self.d2(torch.cat([up_to(d, f2), f2], 1))
        d = self.d1(torch.cat([up_to(d, f1), f1], 1))
        d = self.d0(F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False))
        d = self.dfull(torch.cat([up_to(d, stem), stem], 1))
        return self.head(d)


def loss_fn(logits, target, weight):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce = (bce * weight).mean()
    prob = torch.sigmoid(logits)
    num = 2.0 * (prob * target).sum(dim=(2, 3)) + 1.0
    den = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1.0
    return bce + (1.0 - (num / den).mean())


_m = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE).eval()
with torch.no_grad(), amp_autocast():
    _o = _m(torch.zeros(1, 3, PAD_H, PAD_W, device=DEVICE))
print("model forward ok:", tuple(_o.shape), "| params",
      f"{sum(p.numel() for p in _m.parameters()) / 1e6:.1f}M")
assert _o.shape[-2:] == (PAD_H, PAD_W), "Decoder does not return full resolution."
del _m, _o; gc.collect(); torch.cuda.empty_cache()

model forward ok: (1, 2, 544, 704) | params 32.0M


In [9]:
# ── 9 · Phase A · SimSiam domain adaptation on the provided unlabeled images ────
SSL_ENCODER_PATH = CKPT / "ssl_encoder.pt"


class SimSiam(nn.Module):
    def __init__(self, encoder_name):
        super().__init__()
        self.encoder = timm.create_model(encoder_name, pretrained=True, features_only=True, in_chans=3)
        dim = self.encoder.feature_info.channels()[-1]
        h, z = CFG.SSL_HIDDEN_DIM, CFG.SSL_PROJ_DIM
        self.projector = nn.Sequential(
            nn.Linear(dim, h, bias=False), nn.BatchNorm1d(h), nn.ReLU(inplace=True),
            nn.Linear(h, h, bias=False), nn.BatchNorm1d(h), nn.ReLU(inplace=True),
            nn.Linear(h, z, bias=False), nn.BatchNorm1d(z, affine=False))
        self.predictor = nn.Sequential(
            nn.Linear(z, z // 2, bias=False), nn.BatchNorm1d(z // 2),
            nn.ReLU(inplace=True), nn.Linear(z // 2, z))

    def embed(self, x):
        f = self.encoder(x)[-1]
        return F.adaptive_avg_pool2d(f, 1).flatten(1)

    def forward(self, v1, v2):
        z = self.projector(self.embed(torch.cat([v1, v2], 0)))
        z1, z2 = z.chunk(2, 0)
        return self.predictor(z1), self.predictor(z2), z1.detach(), z2.detach()


class SSLViews(Dataset):
    def __init__(self, paths, size):
        self.paths = list(paths)
        self.size = int(size)

    def __len__(self):
        return len(self.paths)

    def view(self, arr):
        h, w = arr.shape
        s = self.size
        scale = random.uniform(0.5, 1.0)
        ch, cw = min(h, int(h * scale)), min(w, int(w * scale))
        y, x = random.randint(0, h - ch), random.randint(0, w - cw)
        crop = arr[y:y + ch, x:x + cw]
        if HAVE_CV2:
            crop = cv2.resize(crop, (s, s), interpolation=cv2.INTER_LINEAR)
        else:
            crop = np.asarray(Image.fromarray(crop).resize((s, s), Image.BILINEAR))
        v = crop.astype(np.float32) / 255.0
        if random.random() < 0.5:
            v = v[:, ::-1]
        if random.random() < 0.5:
            v = v[::-1]
        v = np.ascontiguousarray(v)
        v = (v - MEAN) * (1.0 + random.uniform(-0.3, 0.3)) + MEAN + random.uniform(-0.2, 0.2) * STD
        if random.random() < 0.3 and HAVE_CV2:
            v = cv2.GaussianBlur(v, (5, 5), random.uniform(0.3, 1.5))
        v = v + np.random.randn(s, s).astype(np.float32) * 0.03 * STD
        v = np.clip(v, 0.0, 1.0)
        t = torch.from_numpy(np.ascontiguousarray(v)).unsqueeze(0).repeat(3, 1, 1)
        return normalize_batch(t)

    def __getitem__(self, i):
        arr = load_gray_u8(self.paths[i])
        return self.view(arr), self.view(arr)


ssl_paths = list(UNLABELED_PATHS)
if CFG.SSL_INCLUDE_TRAIN_IMAGES:
    ssl_paths += [TRAIN_DIR / f"{i}.png" for i in TRAIN_IDS]
ssl_paths += [TEST_DIR / f"{i}.png" for i in TEST_IDS]  # test images are provided, unlabeled, and in-domain
print("SSL corpus:", len(ssl_paths), "images")

if CFG.RESUME and SSL_ENCODER_PATH.exists():
    blob = torch.load(SSL_ENCODER_PATH, map_location="cpu")
    if blob.get("encoder_name") == CFG.ENCODER:
        print("reusing cached SSL encoder:", SSL_ENCODER_PATH,
              "| epochs", blob.get("epochs"), "| final loss", round(blob.get("loss", float("nan")), 4))
        CFG.RUN_SSL = False
    else:
        print("cached SSL encoder is for a different backbone - retraining")

if CFG.RUN_SSL:
    seed_everything(CFG.SPLIT_SEED)
    ssl_model = SimSiam(CFG.ENCODER).to(DEVICE)
    ssl_loader = DataLoader(SSLViews(ssl_paths, CFG.SSL_SIZE), batch_size=CFG.SSL_BATCH, shuffle=True,
                            num_workers=2, pin_memory=True, drop_last=True, persistent_workers=True)
    opt = torch.optim.AdamW(ssl_model.parameters(), lr=CFG.SSL_LR, weight_decay=CFG.SSL_WD)
    total_steps = max(1, CFG.SSL_EPOCHS * len(ssl_loader))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)
    scaler = make_scaler()
    t0, step, last_loss = time.time(), 0, float("nan")
    for ep in range(CFG.SSL_EPOCHS):
        ssl_model.train()
        run = 0.0
        for v1, v2 in ssl_loader:
            if time.time() - t0 > CFG.SSL_BUDGET_SEC:
                break
            v1, v2 = v1.to(DEVICE, non_blocking=True), v2.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                p1, p2, z1, z2 = ssl_model(v1, v2)
                loss = -0.5 * (F.cosine_similarity(p1, z2, dim=1).mean()
                               + F.cosine_similarity(p2, z1, dim=1).mean())
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); step += 1
        last_loss = run / max(1, len(ssl_loader))
        print(f"  ssl epoch {ep + 1:02d}/{CFG.SSL_EPOCHS} loss {last_loss:+.4f} "
              f"({time.time() - t0:.0f}s)")
        if time.time() - t0 > CFG.SSL_BUDGET_SEC:
            print("  ssl budget reached - stopping early (partial adaptation still helps)")
            break
    torch.save({"encoder_name": CFG.ENCODER, "epochs": ep + 1, "loss": float(last_loss),
                "state_dict": ssl_model.encoder.state_dict(), "run_tag": CFG.RUN_TAG},
               SSL_ENCODER_PATH)
    STATS["ssl_epochs"] = ep + 1
    STATS["ssl_loss"] = float(last_loss)
    print("saved:", SSL_ENCODER_PATH, "| elapsed", fmt(time.time() - t0))
    del ssl_model, ssl_loader, opt, scaler
    gc.collect(); torch.cuda.empty_cache()

SSL_BLOB = torch.load(SSL_ENCODER_PATH, map_location="cpu")
print("SSL encoder ready | total elapsed", fmt(elapsed()))

SSL corpus: 2578 images
reusing cached SSL encoder: /kaggle/working/ckpt/ssl_encoder.pt | epochs 25 | final loss -0.9884
SSL encoder ready | total elapsed 0h 00m 31s


In [10]:
# ── 10 · Decode, metric, and fast threshold evaluation ─────────────────────────
def watershed_labels(fg, seed, t_cell, t_seed):
    mask = fg >= t_cell
    if not mask.any():
        return np.zeros(fg.shape, np.int32), 0
    markers, n = ndi.label((seed >= t_seed) & mask)
    if n == 0:
        markers, n = ndi.label(mask)
        if n == 0:
            return np.zeros(fg.shape, np.int32), 0
    lab = watershed(-fg, markers, mask=mask).astype(np.int32)
    return lab, int(lab.max())


def label_stats(lab, fg, seed, nlab):
    flat = lab.ravel()
    area = np.bincount(flat, minlength=nlab + 1).astype(np.float64)
    denom = np.maximum(area, 1.0)
    mean_fg = np.bincount(flat, weights=fg.ravel(), minlength=nlab + 1) / denom
    mean_seed = np.bincount(flat, weights=seed.ravel(), minlength=nlab + 1) / denom
    return area, mean_fg, mean_seed


def prepare_eval(fg, seed, gt_flats, t_cell, t_seed):
    # Everything that depends on (t_cell, t_seed) is computed once; min_area and
    # score_thr are pure post-filters evaluated afterwards for free.
    lab, nlab = watershed_labels(fg, seed, t_cell, t_seed)
    area, mean_fg, mean_seed = label_stats(lab, fg, seed, nlab)
    n_gt = len(gt_flats)
    if nlab == 0 or n_gt == 0:
        return dict(n_gt=n_gt, area=area, score=mean_seed, gi=np.empty(0, int),
                    pi=np.empty(0, int), lab=lab, nlab=nlab)
    flatlab = lab.ravel()
    gt_area = np.array([f.size for f in gt_flats], np.float64)
    C = np.zeros((n_gt, nlab + 1), np.int64)
    for i, idx in enumerate(gt_flats):
        C[i] = np.bincount(flatlab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    return dict(n_gt=n_gt, area=area, score=mean_seed, gi=gi[o], pi=pi[o], lab=lab, nlab=nlab)


def f1_filtered(prep, min_area, score_thr):
    # Official metric: per-image 2TP / (2TP + FP + FN), greedy 1-to-1 at Dice >= 0.50.
    keep = (prep["area"] >= min_area) & (prep["score"] >= score_thr)
    if keep.size:
        keep[0] = False
    n_pred = int(keep.sum())
    n_gt = prep["n_gt"]
    if n_gt == 0:
        return 1.0 if n_pred == 0 else 0.0
    if n_pred == 0:
        return 0.0
    used_g = np.zeros(n_gt, bool)
    used_p = np.zeros(keep.size, bool)
    tp = 0
    for g, p in zip(prep["gi"], prep["pi"]):
        if not keep[p] or used_g[g] or used_p[p]:
            continue
        used_g[g] = True; used_p[p] = True; tp += 1
    return 2.0 * tp / (2.0 * tp + (n_pred - tp) + (n_gt - tp))


def instances_from_maps(fg, seed, post):
    lab, nlab = watershed_labels(fg, seed, post["t_cell"], post["t_seed"])
    if nlab == 0:
        return []
    area, _, mean_seed = label_stats(lab, fg, seed, nlab)
    keep = (area >= post["min_area"]) & (mean_seed >= post.get("score_thr", 0.0))
    keep[0] = False
    flat = lab.ravel()
    order = np.argsort(flat, kind="stable")
    sorted_lab = flat[order]
    bounds = np.searchsorted(sorted_lab, np.arange(nlab + 2))
    out = []
    for j in np.nonzero(keep)[0]:
        idx = order[bounds[j]:bounds[j + 1]]
        if idx.size:
            out.append(np.sort(idx))
    return out


# sanity: the metric must return 1.0 when predictions are the ground truth itself
_vid = val_ids[0]
_gt = INST[_vid]
_fg = TARGETS[_vid][0].astype(np.float32)
_seed = TARGETS[_vid][1].astype(np.float32)
_prep = prepare_eval(_fg, _seed, [f for f in _gt], 0.5, 0.5)
print(f"metric self-check on {_vid}: gt instances {len(_gt)}, decoded {_prep['nlab']}, "
      f"F1 vs GT = {f1_filtered(_prep, 0, 0.0):.4f}  (should be high, not exactly 1.0 - "
      f"watershed on binary targets cannot recover overlapping GT pixels)")
del _prep

metric self-check on 0323e81d23d9: gt instances 25, decoded 25, F1 vs GT = 1.0000  (should be high, not exactly 1.0 - watershed on binary targets cannot recover overlapping GT pixels)


In [11]:
# ── 11 · Inference with flip + scale TTA ───────────────────────────────────────
@torch.no_grad()
def predict_maps(model, ids, image_dir, scales=None, flips=None):
    model.eval()
    scales = CFG.TTA_SCALES if scales is None else scales
    flips = CFG.TTA_FLIPS if flips is None else flips
    out = {}
    for image_id in ids:
        arr = (IMAGES[image_id] if image_dir is None else load_gray_u8(image_dir / f"{image_id}.png"))
        base = arr.astype(np.float32) / 255.0
        acc = np.zeros((2, CFG.IMG_H, CFG.IMG_W), np.float32)
        n = 0
        for s in scales:
            if abs(s - 1.0) < 1e-3:
                img = base
            elif HAVE_CV2:
                img = cv2.resize(base, (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))),
                                 interpolation=cv2.INTER_LINEAR)
            else:
                img = np.asarray(Image.fromarray((base * 255).astype(np.uint8)).resize(
                    (int(round(CFG.IMG_W * s)), int(round(CFG.IMG_H * s))), Image.BILINEAR), np.float32) / 255.0
            h, w = img.shape
            ph = int(math.ceil(h / CFG.PAD_MULT) * CFG.PAD_MULT)
            pw = int(math.ceil(w / CFG.PAD_MULT) * CFG.PAD_MULT)
            padded = pad_to(img, ph, pw)
            x = torch.from_numpy(np.ascontiguousarray(padded)).unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)
            x = normalize_batch(x).to(DEVICE, non_blocking=True)
            variants = [(False, False)] if not flips else [(False, False), (True, False), (False, True), (True, True)]
            for hf, vf in variants:
                xin = x
                if hf:
                    xin = torch.flip(xin, dims=[3])
                if vf:
                    xin = torch.flip(xin, dims=[2])
                with amp_autocast():
                    logits = model(xin)
                prob = torch.sigmoid(logits.float())
                if vf:
                    prob = torch.flip(prob, dims=[2])
                if hf:
                    prob = torch.flip(prob, dims=[3])
                p = prob[0, :, :h, :w].cpu().numpy()
                if p.shape[1:] != (CFG.IMG_H, CFG.IMG_W):
                    if HAVE_CV2:
                        p = np.stack([cv2.resize(p[c], (CFG.IMG_W, CFG.IMG_H),
                                                 interpolation=cv2.INTER_LINEAR) for c in range(2)])
                    else:
                        p = np.stack([np.asarray(Image.fromarray((p[c] * 255).astype(np.uint8)).resize(
                            (CFG.IMG_W, CFG.IMG_H), Image.BILINEAR), np.float32) / 255.0 for c in range(2)])
                acc += p
                n += 1
        out[image_id] = np.clip(acc / max(n, 1) * 255.0, 0, 255).astype(np.uint8)
    return out


def save_maps(tag, split, maps):
    path = MAPS / f"{tag}_{split}.npz"
    np.savez_compressed(path, **maps)
    return path


def load_maps(tag, split):
    with np.load(MAPS / f"{tag}_{split}.npz") as z:
        return {k: z[k] for k in z.files}


def maps_to_float(u8):
    return u8[0].astype(np.float32) / 255.0, u8[1].astype(np.float32) / 255.0


print("inference helpers ready")

inference helpers ready


In [12]:
# ── 12 · Phase B · train one member ────────────────────────────────────────────
def quick_val_f1(model, ids, post):
    maps = predict_maps(model, ids, None, scales=(1.0,), flips=False)
    scores = []
    for image_id in ids:
        fg, seed = maps_to_float(maps[image_id])
        prep = prepare_eval(fg, seed, INST[image_id], post["t_cell"], post["t_seed"])
        scores.append(f1_filtered(prep, post["min_area"], post["score_thr"]))
    return float(np.mean(scores))


def train_member(spec):
    tag = f"{CFG.RUN_TAG}_{spec['name']}"
    best_path = CKPT / f"{tag}_best.pt"
    if CFG.RESUME and best_path.exists():
        blob = torch.load(best_path, map_location="cpu")
        if blob.get("run_tag") == CFG.RUN_TAG and blob.get("member") == spec["name"]:
            print(f"[{spec['name']}] found finished checkpoint (epoch {blob['epoch']}, "
                  f"F1 {blob['f1']:.4f}) - skipping training")
            return blob["f1"]
        print(f"[{spec['name']}] checkpoint belongs to another run ({blob.get('run_tag')}) - retraining")

    seed_everything(spec["seed"])
    model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=True).to(DEVICE)
    missing = model.encoder.load_state_dict(SSL_BLOB["state_dict"], strict=False)
    print(f"[{spec['name']}] SSL encoder loaded | missing {len(missing.missing_keys)} "
          f"unexpected {len(missing.unexpected_keys)}")

    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW(
        [{"params": dec_params, "lr": CFG.SUP_LR},
         {"params": enc_params, "lr": CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT}],
        weight_decay=CFG.SUP_WD)

    loader = DataLoader(CellDataset(train_ids, spec["crop"], True), batch_size=CFG.SUP_BATCH,
                        shuffle=True, num_workers=2, pin_memory=True, drop_last=True,
                        persistent_workers=True)
    total_steps = CFG.SUP_EPOCHS * CFG.STEPS_PER_EPOCH
    scaler = make_scaler()
    best_f1, best_epoch, log = -1.0, -1, []
    t0 = time.time()

    def lr_at(step):
        if step < CFG.WARMUP_STEPS:
            return (step + 1) / max(1, CFG.WARMUP_STEPS)
        p = (step - CFG.WARMUP_STEPS) / max(1, total_steps - CFG.WARMUP_STEPS)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

    step = 0
    it = iter(loader)
    for ep in range(CFG.SUP_EPOCHS):
        if time.time() - t0 > CFG.MEMBER_BUDGET_SEC:
            print(f"[{spec['name']}] member budget reached at epoch {ep} - stopping")
            break
        freeze = ep < CFG.FREEZE_ENCODER_EPOCHS
        for p in enc_params:
            p.requires_grad_(not freeze)
        model.train()
        run = 0.0
        for _ in range(CFG.STEPS_PER_EPOCH):
            try:
                x, y, w, _ = next(it)
            except StopIteration:
                it = iter(loader)
                x, y, w, _ = next(it)
            x = normalize_batch(x).to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            w = w.to(DEVICE, non_blocking=True)
            mult = lr_at(step)
            opt.param_groups[0]["lr"] = CFG.SUP_LR * mult
            opt.param_groups[1]["lr"] = CFG.SUP_LR * CFG.SUP_ENCODER_LR_MULT * mult * (0.0 if freeze else 1.0)
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = loss_fn(model(x), y, w)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update()
            run += loss.item(); step += 1
        msg = f"[{spec['name']}] ep {ep + 1:02d}/{CFG.SUP_EPOCHS} loss {run / CFG.STEPS_PER_EPOCH:.4f}"
        if ep + 1 >= CFG.EVAL_START and (ep + 1 - CFG.EVAL_START) % CFG.EVAL_EVERY == 0:
            f1 = quick_val_f1(model, val_ids, CFG.DEFAULT_POST)
            msg += f" | val F1 {f1:.4f}"
            if f1 > best_f1:
                best_f1, best_epoch = f1, ep + 1
                torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": ep + 1,
                            "f1": f1, "state_dict": model.state_dict(), "spec": spec}, best_path)
                msg += "  *"
            log.append(dict(member=spec["name"], epoch=ep + 1, loss=run / CFG.STEPS_PER_EPOCH, f1=f1))
        print(msg + f" | {fmt(time.time() - t0)}")

    if best_f1 < 0:  # never evaluated (budget or smoke test) - keep the last weights
        best_f1, best_epoch = quick_val_f1(model, val_ids, CFG.DEFAULT_POST), CFG.SUP_EPOCHS
        torch.save({"run_tag": CFG.RUN_TAG, "member": spec["name"], "epoch": best_epoch,
                    "f1": best_f1, "state_dict": model.state_dict(), "spec": spec}, best_path)
    pd.DataFrame(log).to_csv(ART / f"train_log_{spec['name']}.csv", index=False)
    print(f"[{spec['name']}] best epoch {best_epoch} | quick val F1 {best_f1:.4f} | {fmt(time.time() - t0)}")
    del model, opt, loader, scaler
    gc.collect(); torch.cuda.empty_cache()
    return best_f1

In [13]:
# ── 13 · Phase B/C · run members, cache maps, write a per-member submission ────
def write_submission(path, maps, ids, post, cluster_post=None):
    rows, counts, fallbacks = [], [], 0
    for image_id in ids:
        fg, seed = maps_to_float(maps[image_id])
        use = post
        if cluster_post is not None:
            use = cluster_post.get(CLUSTER.get(image_id, 0), post)
        inst = instances_from_maps(fg, seed, use)
        if not inst:  # an image with GT and zero masks scores exactly 0.0 - never allow it
            fallbacks += 1
            inst = instances_from_maps(fg, seed, dict(t_cell=0.30, t_seed=0.30,
                                                      min_area=0, score_thr=0.0))[:1]
            if not inst:
                h = CFG.IMG_H // 2
                inst = [np.arange(h * CFG.IMG_W, h * CFG.IMG_W + 25)]
        counts.append(len(inst))
        rows.extend((image_id, flat_to_rle(f)) for f in inst)
    frame = pd.DataFrame(rows, columns=[ID_COL, ANN_COL])
    frame.insert(0, CFG.SUB_INDEX_COL, np.arange(len(frame), dtype=np.int64))
    frame.to_csv(path, index=False)
    assert set(frame[ID_COL].astype(str)) == set(ids), "Submission is missing test images."
    assert frame[ANN_COL].notna().all() and frame[CFG.SUB_INDEX_COL].is_unique
    return dict(path=str(path), rows=len(frame), mean=float(np.mean(counts)),
                min=int(np.min(counts)), max=int(np.max(counts)), fallbacks=fallbacks)


MEMBER_RESULTS = []
for spec in CFG.MEMBERS:
    if elapsed() > CFG.TOTAL_BUDGET_SEC - CFG.FINAL_RESERVE_SEC:
        print(f"!! stopping after {len(MEMBER_RESULTS)} member(s) - reserving time for the sweep")
        break
    tag = f"{CFG.RUN_TAG}_{spec['name']}"
    quick = train_member(spec)

    if CFG.RESUME and (MAPS / f"{tag}_val.npz").exists() and (MAPS / f"{tag}_test.npz").exists():
        print(f"[{spec['name']}] cached maps found - skipping inference")
    else:
        blob = torch.load(CKPT / f"{tag}_best.pt", map_location="cpu")
        assert blob["run_tag"] == CFG.RUN_TAG and blob["member"] == spec["name"], \
            "Checkpoint identity mismatch - refusing to build maps from another run."
        model = UNet(CFG.ENCODER, CFG.OUT_CHANNELS, CFG.DECODER_CHANNELS, pretrained=False).to(DEVICE)
        model.load_state_dict(blob["state_dict"])
        t0 = time.time()
        save_maps(tag, "val", predict_maps(model, val_ids, None))
        save_maps(tag, "test", predict_maps(model, TEST_IDS, TEST_DIR))
        print(f"[{spec['name']}] TTA maps cached in {fmt(time.time() - t0)}")
        del model, blob
        gc.collect(); torch.cuda.empty_cache()

    # A submittable CSV exists from this point on, whatever happens next.
    vmaps = load_maps(tag, "val")
    tta_f1 = float(np.mean([
        f1_filtered(prepare_eval(*maps_to_float(vmaps[i]), INST[i],
                                 CFG.DEFAULT_POST["t_cell"], CFG.DEFAULT_POST["t_seed"]),
                    CFG.DEFAULT_POST["min_area"], CFG.DEFAULT_POST["score_thr"])
        for i in val_ids]))
    info = write_submission(ART / f"submission_{spec['name']}.csv", load_maps(tag, "test"),
                            TEST_IDS, CFG.DEFAULT_POST)
    print(f"[{spec['name']}] TTA val F1 (default post) {tta_f1:.4f} | "
          f"test preds/image {info['mean']:.1f} | {info['path']}")
    MEMBER_RESULTS.append(dict(member=spec["name"], quick_f1=quick, tta_f1=tta_f1, **info))
    del vmaps
    gc.collect()

STATS["members"] = MEMBER_RESULTS
print("\nmembers complete:", [m["member"] for m in MEMBER_RESULTS], "| elapsed", fmt(elapsed()))
assert MEMBER_RESULTS, "No member finished - nothing to ensemble."

[m1] found finished checkpoint (epoch 36, F1 0.7745) - skipping training
[m1] cached maps found - skipping inference
[m1] TTA val F1 (default post) 0.7999 | test preds/image 115.2 | /kaggle/working/artifacts/submission_m1.csv
[m2] found finished checkpoint (epoch 46, F1 0.7719) - skipping training
[m2] cached maps found - skipping inference
[m2] TTA val F1 (default post) 0.7938 | test preds/image 112.5 | /kaggle/working/artifacts/submission_m2.csv
[m3] found finished checkpoint (epoch 34, F1 0.7739) - skipping training
[m3] cached maps found - skipping inference
[m3] TTA val F1 (default post) 0.7866 | test preds/image 109.0 | /kaggle/working/artifacts/submission_m3.csv

members complete: ['m1', 'm2', 'm3'] | elapsed 0h 01m 26s


In [ ]:
# ── 14 · Phase D · ensemble the validation maps and sweep the decode ───────────
ACTIVE = [m["member"] for m in MEMBER_RESULTS]
print("averaging validation maps over:", ACTIVE)

ENS_VAL = None
for name in ACTIVE:
    mp = load_maps(f"{CFG.RUN_TAG}_{name}", "val")
    if ENS_VAL is None:
        ENS_VAL = {k: mp[k].astype(np.float32) for k in mp}
    else:
        for k in ENS_VAL:
            ENS_VAL[k] += mp[k].astype(np.float32)
    del mp
    gc.collect()
for k in ENS_VAL:
    ENS_VAL[k] = ENS_VAL[k] / len(ACTIVE) / 255.0

sub_ids = val_ids[:CFG.SWEEP_SUBSET] if len(val_ids) > CFG.SWEEP_SUBSET else val_ids


def sweep(image_ids, t_cell_grid, t_seed_grid, verbose=True):
    rows = []
    per_image = {}
    t0 = time.time()
    for tc in t_cell_grid:
        for ts in t_seed_grid:
            preps = [prepare_eval(ENS_VAL[i][0], ENS_VAL[i][1], INST[i], tc, ts) for i in image_ids]
            for ma in CFG.MIN_AREA_GRID:
                for st in CFG.SCORE_THR_GRID:
                    vals = np.array([f1_filtered(p, ma, st) for p in preps])
                    rows.append(dict(t_cell=tc, t_seed=ts, min_area=ma, score_thr=st,
                                     f1=float(vals.mean()),
                                     mean_pred=float(np.mean([
                                         int(((p["area"][1:] >= ma) & (p["score"][1:] >= st)).sum())
                                         for p in preps]))))
                    per_image[(tc, ts, ma, st)] = vals
            del preps
        if verbose:
            print(f"  t_cell {tc:.2f} done ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True), per_image


print(f"coarse sweep on {len(sub_ids)} images, "
      f"{len(CFG.T_CELL_GRID) * len(CFG.T_SEED_GRID)} watershed settings")
coarse, _ = sweep(sub_ids, CFG.T_CELL_GRID, CFG.T_SEED_GRID)
print(coarse.head(10).to_string(index=False))

top = coarse.head(CFG.SWEEP_REFINE_TOP)
refine_tc = sorted(set(top.t_cell.tolist()))
refine_ts = sorted(set(top.t_seed.tolist()))
print(f"\nrefining on all {len(val_ids)} validation images | t_cell {refine_tc} t_seed {refine_ts}")
fine, PER_IMAGE = sweep(val_ids, refine_tc, refine_ts)

# Grid-edge guard: this is exactly the trap that capped Stages 5 and 6 at t_seed = 0.65.
best = fine.iloc[0]
if abs(float(best.t_seed) - max(CFG.T_SEED_GRID)) < 1e-9 and CFG.T_SEED_EXTENSION:
    print(f"\n!! optimum t_seed = {best.t_seed:.2f} sits on the grid edge - extending to "
          f"{CFG.T_SEED_EXTENSION}")
    ext, ext_per = sweep(val_ids, refine_tc, CFG.T_SEED_EXTENSION)
    fine = pd.concat([fine, ext]).sort_values("f1", ascending=False).reset_index(drop=True)
    PER_IMAGE.update(ext_per)
    best = fine.iloc[0]
    if abs(float(best.t_seed) - max(CFG.T_SEED_EXTENSION)) < 1e-9:
        print("!! still on the edge after extension - the seed threshold is not the binding constraint")

fine.to_csv(ART / "sweep_results.csv", index=False)
print("\n" + fine.head(12).to_string(index=False))

# Report whether score_thr does anything at all. In Stage 5 it was perfectly degenerate.
at_best = fine[(fine.t_cell == best.t_cell) & (fine.t_seed == best.t_seed) &
               (fine.min_area == best.min_area)]
print(f"\nscore_thr axis at the optimum: {at_best.f1.nunique()} distinct F1 values over "
      f"{len(at_best)} thresholds -> "
      f"{'USEFUL' if at_best.f1.nunique() > 1 else 'DEGENERATE (mean-seed score does not separate)'}")

In [ ]:
# ── 15 · Plateau-centre selection and per-cluster thresholds ───────────────────
# Never take the peak: pick the centre of the region within ROBUST_TOL of it.
ROBUST_TOL = 0.004
peak = float(fine.f1.max())
plateau = fine[fine.f1 >= peak - ROBUST_TOL]
POST = dict(t_cell=float(np.median(plateau.t_cell)), t_seed=float(np.median(plateau.t_seed)),
            min_area=int(np.median(plateau.min_area)), score_thr=float(np.median(plateau.score_thr)))


def snap(value, grid):
    return min(grid, key=lambda g: abs(g - value))

POST["t_cell"] = snap(POST["t_cell"], sorted(set(fine.t_cell)))
POST["t_seed"] = snap(POST["t_seed"], sorted(set(fine.t_seed)))
POST["min_area"] = int(snap(POST["min_area"], CFG.MIN_AREA_GRID))
POST["score_thr"] = snap(POST["score_thr"], CFG.SCORE_THR_GRID)
key = (POST["t_cell"], POST["t_seed"], POST["min_area"], POST["score_thr"])
if key not in PER_IMAGE:
    POST = dict(t_cell=float(best.t_cell), t_seed=float(best.t_seed),
                min_area=int(best.min_area), score_thr=float(best.score_thr))
    key = (POST["t_cell"], POST["t_seed"], POST["min_area"], POST["score_thr"])
GLOBAL_F1 = float(PER_IMAGE[key].mean())
print(f"peak F1 {peak:.4f} | plateau size {len(plateau)} | chosen (centre) {POST} -> F1 {GLOBAL_F1:.4f}")

# Per-cluster thresholds, gated. Only the refined combos are candidates.
val_cluster_idx = {c: np.array([n for n, i in enumerate(val_ids) if CLUSTER[i] == c])
                   for c in range(CFG.CLUSTER_K)}
cluster_post, cluster_gain = {}, {}
for c, idx in val_cluster_idx.items():
    if idx.size == 0:
        cluster_post[c] = POST
        continue
    scored = [(float(PER_IMAGE[k][idx].mean()), k) for k in PER_IMAGE]
    scored.sort(reverse=True)
    bf1, bk = scored[0]
    cluster_post[c] = dict(t_cell=bk[0], t_seed=bk[1], min_area=int(bk[2]), score_thr=bk[3])
    cluster_gain[c] = bf1 - float(PER_IMAGE[key][idx].mean())
    print(f"  cluster {c} (n={idx.size}): global {PER_IMAGE[key][idx].mean():.4f} -> "
          f"tuned {bf1:.4f} (+{cluster_gain[c]:.4f}) {cluster_post[c]}")

combined = np.zeros(len(val_ids))
for c, idx in val_cluster_idx.items():
    if idx.size == 0:
        continue
    k = (cluster_post[c]["t_cell"], cluster_post[c]["t_seed"],
         cluster_post[c]["min_area"], cluster_post[c]["score_thr"])
    combined[idx] = PER_IMAGE[k][idx]
CLUSTER_F1 = float(combined.mean())
USE_CLUSTER = CLUSTER_F1 >= GLOBAL_F1 + CFG.PER_CLUSTER_MIN_GAIN
print(f"\nper-cluster combined F1 {CLUSTER_F1:.4f} vs global {GLOBAL_F1:.4f} "
      f"-> {'ACCEPTED' if USE_CLUSTER else 'REJECTED (below the +%.3f gate)' % CFG.PER_CLUSTER_MIN_GAIN}")
STATS.update(global_post=POST, global_val_f1=GLOBAL_F1, cluster_val_f1=CLUSTER_F1,
             use_cluster=bool(USE_CLUSTER),
             cluster_post={str(k): v for k, v in cluster_post.items()})

In [ ]:
# ── 16 · Encode/decode parity on the actual predictions ────────────────────────
# Writes the validation predictions as RLE, reads them back, and re-scores. If this
# does not match the number above, the submission pipeline is losing points that
# local validation cannot see.
parity = []
for image_id in val_ids[:25]:
    fg, seed = ENS_VAL[image_id][0], ENS_VAL[image_id][1]
    inst = instances_from_maps(fg, seed, POST)
    if not inst:
        parity.append(0.0)
        continue
    round_trip = [rle_decode_flat(flat_to_rle(f), CFG.IMG_H, CFG.IMG_W, RLE_ORDER) for f in inst]
    lab = np.zeros(NPIX, np.int32)
    for j, f in enumerate(round_trip, 1):
        lab[f] = j
    lab = lab.reshape(CFG.IMG_H, CFG.IMG_W)
    nlab = len(round_trip)
    gt_flats = INST[image_id]
    gt_area = np.array([f.size for f in gt_flats], np.float64)
    area = np.bincount(lab.ravel(), minlength=nlab + 1).astype(np.float64)
    C = np.zeros((len(gt_flats), nlab + 1), np.int64)
    for i, idx in enumerate(gt_flats):
        C[i] = np.bincount(lab.ravel()[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    used_g, used_p, tp = set(), set(), 0
    for g, p in zip(gi[o], pi[o]):
        if g in used_g or p in used_p:
            continue
        used_g.add(g); used_p.add(p); tp += 1
    parity.append(2.0 * tp / (2.0 * tp + (nlab - tp) + (len(gt_flats) - tp)))

direct = float(PER_IMAGE[key][:25].mean())
print(f"parity check on 25 images | in-memory {direct:.4f} | after RLE round-trip {np.mean(parity):.4f} "
      f"| delta {abs(direct - np.mean(parity)):.5f}")
if abs(direct - np.mean(parity)) > 0.002:
    print("!! The submission encoding is losing score. Investigate before submitting.")
else:
    print(">>> encoding is lossless - the local-to-public gap is distribution shift, not a codec bug")
STATS["parity_delta"] = float(abs(direct - np.mean(parity)))
del ENS_VAL
gc.collect()

In [ ]:
# ── 17 · Phase F · ensemble the test maps and write the final submissions ──────
ENS_TEST = None
for name in ACTIVE:
    mp = load_maps(f"{CFG.RUN_TAG}_{name}", "test")
    if ENS_TEST is None:
        ENS_TEST = {k: mp[k].astype(np.float32) for k in mp}
    else:
        for k in ENS_TEST:
            ENS_TEST[k] += mp[k].astype(np.float32)
    del mp
    gc.collect()
for k in ENS_TEST:
    ENS_TEST[k] = np.clip(ENS_TEST[k] / len(ACTIVE), 0, 255).astype(np.uint8)

SUBS = []
info = write_submission(WORK / "submission.csv", ENS_TEST, TEST_IDS, POST)
SUBS.append(dict(name="ensemble_global", local_f1=GLOBAL_F1, **info))
shutil.copy2(WORK / "submission.csv", ART / "submission_ensemble_global.csv")

if USE_CLUSTER:
    info_c = write_submission(ART / "submission_ensemble_cluster.csv", ENS_TEST, TEST_IDS,
                              POST, cluster_post=cluster_post)
    SUBS.append(dict(name="ensemble_per_cluster", local_f1=CLUSTER_F1, **info_c))

for m in MEMBER_RESULTS:
    SUBS.append(dict(name=f"single_{m['member']}", local_f1=m["tta_f1"], path=m["path"],
                     rows=m["rows"], mean=m["mean"], min=m["min"], max=m["max"],
                     fallbacks=m["fallbacks"]))

sub_table = pd.DataFrame(SUBS).sort_values("local_f1", ascending=False)
sub_table.to_csv(ART / "candidate_submissions.csv", index=False)
print(sub_table.to_string(index=False))
print(f"\nGT mean instances/image {STATS['gt_mean_instances']:.1f} | "
      f"ensemble test predictions/image {info['mean']:.1f}")
if info["mean"] < STATS["gt_mean_instances"] * 0.9:
    print("   still under-segmenting relative to the training distribution - "
          "if t_seed hit the ceiling again, the seed target erosion fraction is the next lever")
STATS["submissions"] = SUBS

In [ ]:
# ── 18 · Report and artifact listing ───────────────────────────────────────────
lines = [
    "# Stage 7 report - SSL-adapted ConvNeXt-UNet ensemble",
    f"- run tag: {CFG.RUN_TAG}",
    f"- encoder: {CFG.ENCODER} (+ SimSiam adaptation, {STATS.get('ssl_epochs', 'cached')} epochs "
    f"@ {CFG.SSL_SIZE}px on {len(ssl_paths)} unlabeled images)",
    f"- members: {[m['member'] for m in MEMBER_RESULTS]} | fold {CFG.FOLD}/{CFG.N_FOLDS} | "
    f"train/val {len(train_ids)}/{len(val_ids)} | groups {STATS['n_groups']}",
    f"- max val-to-train correlation: {STATS['max_val_train_corr']:.3f}",
    f"- per-member TTA val F1: " + ", ".join(f"{m['member']} {m['tta_f1']:.4f}" for m in MEMBER_RESULTS),
    f"- ensemble val F1 (global thresholds): {GLOBAL_F1:.4f}",
    f"- ensemble val F1 (per-cluster): {CLUSTER_F1:.4f} -> {'used' if USE_CLUSTER else 'rejected'}",
    f"- chosen post-processing: {POST}",
    f"- RLE parity delta: {STATS['parity_delta']:.5f}",
    f"- test predictions/image: mean {info['mean']:.1f} (min {info['min']}, max {info['max']}), "
    f"fallbacks {info['fallbacks']}",
    f"- cluster mixture train {np.round(STATS['cluster_mix_train'], 3).tolist()} vs "
    f"test {np.round(STATS['cluster_mix_test'], 3).tolist()}",
    f"- total elapsed: {fmt(elapsed())}",
    "",
    "## How to read the local number",
    "Stages 2/4/5 fall on a line where public = 0.773 * local - 0.111. Stage 6, whose encoder was",
    "SimSiam-adapted on the unlabeled set, scored 0.033 ABOVE that line. This notebook is also",
    "SSL-adapted, so compare its local F1 against Stage 6's 0.7740 (which produced 0.5209), not",
    "against Stage 5's 0.8279 (which produced 0.5293). A local score near 0.80 here is expected to",
    "beat 0.5293; do not reject it for looking lower than Stage 5.",
    "",
    "## Submit order",
    "1. submission.csv (ensemble, global thresholds) - the primary candidate.",
    "2. submission_ensemble_cluster.csv only if the gate above accepted it.",
    "Keep the Stage 5 file at 0.5293 selected as one of the two final submissions until something",
    "measurably beats it on the public leaderboard.",
]
(ART / "REPORT.md").write_text("\n".join(lines))
with open(ART / "stats.json", "w") as f:
    json.dump(STATS, f, indent=2, default=str)

print("\n".join(lines))
print("\nArtifacts in", ART)
for p in sorted(ART.iterdir()):
    print(f"  {p.name:44s} {p.stat().st_size / 1e6:8.2f} MB")
print("\nPrimary submission:", WORK / "submission.csv")
print("Total elapsed:", fmt(elapsed()))

## Stage 9

In [14]:
# ── S1 · Load the Stage 8 non-SSL maps and prove both families share a split ───
# This is the honest-blend gate. If the validation sets differ by even one image,
# the blend weight is being tuned against a moving target.
STAGE8_TAG = "stage8_nonssl"

S8_DIR = None
for cand in sorted(Path("/kaggle/input").glob("*")):
    if list(cand.rglob(f"{STAGE8_TAG}_*_val.npz")):
        S8_DIR = next(iter(cand.rglob(f"{STAGE8_TAG}_*_val.npz"))).parent
        break
assert S8_DIR is not None, (
    "Stage 8 dataset not attached. Add Input -> Datasets -> stage_8_nonssl.\n"
    f"Currently under /kaggle/input: {[p.name for p in Path('/kaggle/input').glob('*')]}")
print("stage 8 dir:", S8_DIR)

NON_MEMBERS = sorted({p.name.split("_")[2] for p in S8_DIR.glob(f"{STAGE8_TAG}_*_val.npz")})
SSL_MEMBERS = [m for m in ("m1", "m2", "m3") if (MAPS / f"{CFG.RUN_TAG}_{m}_val.npz").exists()]
print("non-SSL members:", NON_MEMBERS, "| SSL members:", SSL_MEMBERS)
assert NON_MEMBERS and SSL_MEMBERS, "Need at least one member from each family to blend."

vid_path = next(iter(S8_DIR.rglob("val_ids.json")), None)
assert vid_path is not None, "val_ids.json missing from the Stage 8 dataset."
s8 = json.loads(vid_path.read_text())
same = list(s8["val_ids"]) == list(val_ids)
print(f"stage 8 split: train {s8['n_train']} / val {len(s8['val_ids'])} / groups {s8['n_groups']}")
print(f"stage 7 split: train {len(train_ids)} / val {len(val_ids)} / groups {STATS['n_groups']}")
if not same:
    only8 = set(s8["val_ids"]) - set(val_ids)
    only7 = set(val_ids) - set(s8["val_ids"])
    raise RuntimeError(
        f"VALIDATION SETS DIFFER - the blend cannot be tuned honestly.\n"
        f"  only in stage 8: {sorted(only8)[:6]} ({len(only8)} total)\n"
        f"  only in stage 7: {sorted(only7)[:6]} ({len(only7)} total)\n"
        "Submit the two families separately instead of blending them.")
print(">>> identical validation sets - the blend weight is tunable.")


def load_maps_any(member, split):
    if member in SSL_MEMBERS:
        with np.load(MAPS / f"{CFG.RUN_TAG}_{member}_{split}.npz") as z:
            return {k: z[k] for k in z.files}
    with np.load(S8_DIR / f"{STAGE8_TAG}_{member}_{split}.npz") as z:
        return {k: z[k] for k in z.files}


def family_mean(members, split, ids, as_float=True):
    acc = None
    for m in members:
        mp = load_maps_any(m, split)
        if acc is None:
            acc = {i: mp[i].astype(np.float32) for i in ids}
        else:
            for i in ids:
                acc[i] += mp[i].astype(np.float32)
        del mp
        gc.collect()
    for i in ids:
        acc[i] /= len(members)
        if as_float:
            acc[i] /= 255.0
    return acc

stage 8 dir: /kaggle/input/datasets/ghostylicious/stage-8-nonssl
non-SSL members: ['n1', 'n2'] | SSL members: ['m1', 'm2', 'm3']
stage 8 split: train 338 / val 86 / groups 421
stage 7 split: train 338 / val 86 / groups 421
>>> identical validation sets - the blend weight is tunable.


In [15]:
# ── S2 · Sweep the blend weight, then refine the decode at the winner ──────────
# w = 0.0 is pure SSL (Stage 7), w = 1.0 is pure non-SSL (Stage 8). Both endpoints
# are in the grid so the blend has to earn its place rather than be assumed.
W_GRID = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]

# score_thr was degenerate in Stage 5 and chose 0.0 on its own in Stage 7 - axis dropped.
S9_SCORE_THR = [0.0]
COARSE_TC = [0.45, 0.50, 0.55]
COARSE_TS = [0.55, 0.60, 0.65, 0.70]
FINE_TC = [0.40, 0.45, 0.50, 0.55, 0.60]
FINE_TS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
FINE_MA = [30, 50, 70, 90, 110]
COARSE_N = 45

print("building family means on validation ...", flush=True)
SSL_VAL = family_mean(SSL_MEMBERS, "val", val_ids)
NON_VAL = family_mean(NON_MEMBERS, "val", val_ids)
print("ready | elapsed", fmt(elapsed()))


def blend(a, b, w, ids):
    # w * non-SSL + (1 - w) * SSL
    return {i: (w * b[i] + (1.0 - w) * a[i]) for i in ids}


def sweep9(maps, ids, tc_grid, ts_grid, ma_grid, label=""):
    rows, per_image = [], {}
    t0 = time.time()
    for tc in tc_grid:
        for ts in ts_grid:
            preps = [prepare_eval(maps[i][0], maps[i][1], INST[i], tc, ts) for i in ids]
            for ma in ma_grid:
                for st in S9_SCORE_THR:
                    vals = np.array([f1_filtered(p, ma, st) for p in preps])
                    rows.append(dict(t_cell=tc, t_seed=ts, min_area=ma, score_thr=st,
                                     f1=float(vals.mean()),
                                     mean_pred=float(np.mean([
                                         int(((p["area"][1:] >= ma) & (p["score"][1:] >= st)).sum())
                                         for p in preps]))))
                    per_image[(tc, ts, ma, st)] = vals
            del preps
    print(f"  {label} done in {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True), per_image


coarse_ids = val_ids[:COARSE_N]
w_rows = []
for w in W_GRID:
    bl = blend(SSL_VAL, NON_VAL, w, coarse_ids)
    tab, _ = sweep9(bl, coarse_ids, COARSE_TC, COARSE_TS, [50, 90], label=f"w={w:.2f}")
    best = tab.iloc[0]
    w_rows.append(dict(w=w, f1=float(best.f1), t_cell=float(best.t_cell),
                       t_seed=float(best.t_seed), min_area=int(best.min_area),
                       mean_pred=float(best.mean_pred)))
    print(f"  w={w:.2f} coarse F1 {best.f1:.4f} (preds/img {best.mean_pred:.1f})")
    del bl
    gc.collect()

wtab = pd.DataFrame(w_rows).sort_values("f1", ascending=False).reset_index(drop=True)
print("\n" + wtab.to_string(index=False))
BEST_W = float(wtab.iloc[0].w)
print(f"\n>>> best blend weight w = {BEST_W:.2f}  "
      f"({'pure SSL' if BEST_W == 0 else 'pure non-SSL' if BEST_W == 1 else 'genuine blend'})")

print(f"\nrefining on all {len(val_ids)} validation images at w = {BEST_W:.2f} ...", flush=True)
BL_VAL = blend(SSL_VAL, NON_VAL, BEST_W, val_ids)
fine9, PER9 = sweep9(BL_VAL, val_ids, FINE_TC, FINE_TS, FINE_MA, label="refine")
fine9.to_csv(ART / "stage9_sweep.csv", index=False)
print(fine9.head(10).to_string(index=False))

# Plateau centre, never the peak - a 86-image optimum is not a location, it is a region.
peak9 = float(fine9.f1.max())
plat9 = fine9[fine9.f1 >= peak9 - 0.004]
snap = lambda v, grid: min(grid, key=lambda g: abs(g - v))
POST9 = dict(t_cell=snap(float(np.median(plat9.t_cell)), FINE_TC),
             t_seed=snap(float(np.median(plat9.t_seed)), FINE_TS),
             min_area=int(snap(int(np.median(plat9.min_area)), FINE_MA)),
             score_thr=0.0)
KEY9 = (POST9["t_cell"], POST9["t_seed"], POST9["min_area"], POST9["score_thr"])
if KEY9 not in PER9:
    b = fine9.iloc[0]
    POST9 = dict(t_cell=float(b.t_cell), t_seed=float(b.t_seed),
                 min_area=int(b.min_area), score_thr=0.0)
    KEY9 = (POST9["t_cell"], POST9["t_seed"], POST9["min_area"], POST9["score_thr"])
GLOBAL9 = float(PER9[KEY9].mean())
print(f"\nplateau centre {POST9} | local F1 {GLOBAL9:.4f} (peak {peak9:.4f})")
print(f"reference: Stage 7 ensemble 0.8035 local -> 0.5259 public | Stage 5 0.8279 -> 0.5293")
del SSL_VAL, NON_VAL
gc.collect()

building family means on validation ...
ready | elapsed 0h 01m 56s
  w=0.00 done in 13s
  w=0.00 coarse F1 0.8102 (preds/img 117.7)
  w=0.30 done in 13s
  w=0.30 coarse F1 0.8125 (preds/img 117.3)
  w=0.40 done in 13s
  w=0.40 coarse F1 0.8134 (preds/img 116.7)
  w=0.50 done in 13s
  w=0.50 coarse F1 0.8144 (preds/img 116.8)
  w=0.60 done in 13s
  w=0.60 coarse F1 0.8143 (preds/img 116.9)
  w=0.70 done in 13s
  w=0.70 coarse F1 0.8142 (preds/img 116.0)
  w=1.00 done in 13s
  w=1.00 coarse F1 0.8104 (preds/img 116.6)

  w       f1  t_cell  t_seed  min_area  mean_pred
0.5 0.814394    0.50    0.65        90 116.800000
0.6 0.814305    0.50    0.70        90 116.911111
0.7 0.814153    0.55    0.70        90 115.977778
0.4 0.813390    0.50    0.65        90 116.733333
0.3 0.812483    0.45    0.65        90 117.311111
1.0 0.810416    0.55    0.70        90 116.622222
0.0 0.810157    0.45    0.65        90 117.688889

>>> best blend weight w = 0.50  (genuine blend)

refining on all 86 validati

0

In [16]:
# ── S3 · Per-density-bin thresholds (gate: +0.005) ────────────────────────────
# The density proxy is the predicted foreground fraction, so it is computable on the
# test images too - no ground truth needed. Tertile edges are fixed on validation and
# reused unchanged on test.
def density_of(maps, ids):
    return np.array([float((maps[i][0] >= 0.5).mean()) for i in ids])

dens_val = density_of(BL_VAL, val_ids)
EDGES = np.quantile(dens_val, [1 / 3, 2 / 3])
bin_val = np.digitize(dens_val, EDGES)
print("density tertile edges:", np.round(EDGES, 4),
      "| bin sizes", np.bincount(bin_val, minlength=3).tolist())

dens_post, gains = {}, {}
for b in range(3):
    idx = np.nonzero(bin_val == b)[0]
    if idx.size == 0:
        dens_post[b] = POST9
        continue
    scored = sorted(((float(PER9[k][idx].mean()), k) for k in PER9), reverse=True)
    bf1, bk = scored[0]
    dens_post[b] = dict(t_cell=bk[0], t_seed=bk[1], min_area=int(bk[2]), score_thr=bk[3])
    gains[b] = bf1 - float(PER9[KEY9][idx].mean())
    print(f"  bin {b} (n={idx.size}): global {PER9[KEY9][idx].mean():.4f} -> "
          f"tuned {bf1:.4f} (+{gains[b]:.4f})  {dens_post[b]}")

combined = np.zeros(len(val_ids))
for b in range(3):
    idx = np.nonzero(bin_val == b)[0]
    if idx.size:
        p = dens_post[b]
        combined[idx] = PER9[(p["t_cell"], p["t_seed"], p["min_area"], p["score_thr"])][idx]
DENS9 = float(combined.mean())
USE_DENS = DENS9 >= GLOBAL9 + 0.005
print(f"\ndensity-adaptive F1 {DENS9:.4f} vs global {GLOBAL9:.4f} -> "
      f"{'ACCEPTED' if USE_DENS else 'REJECTED (below the +0.005 gate)'}")
print("  ~29 images per bin, so the gate is doing real work - a small win here is fitting noise.")

density tertile edges: [0.0231 0.163 ] | bin sizes [29, 28, 29]
  bin 0 (n=29): global 0.8904 -> tuned 0.8917 (+0.0013)  {'t_cell': 0.5, 't_seed': 0.75, 'min_area': 70, 'score_thr': 0.0}
  bin 1 (n=28): global 0.7921 -> tuned 0.7970 (+0.0049)  {'t_cell': 0.45, 't_seed': 0.7, 'min_area': 110, 'score_thr': 0.0}
  bin 2 (n=29): global 0.7386 -> tuned 0.7470 (+0.0084)  {'t_cell': 0.6, 't_seed': 0.75, 'min_area': 70, 'score_thr': 0.0}

density-adaptive F1 0.8121 vs global 0.8072 -> REJECTED (below the +0.005 gate)
  ~29 images per bin, so the gate is doing real work - a small win here is fitting noise.


In [17]:
# ── S4 · RLE parity, test blend, write /kaggle/working/submission.csv ──────────
parity = []
for image_id in val_ids[:25]:
    inst = instances_from_maps(BL_VAL[image_id][0], BL_VAL[image_id][1], POST9)
    if not inst:
        parity.append(0.0); continue
    rt = [rle_decode_flat(flat_to_rle(f), CFG.IMG_H, CFG.IMG_W, RLE_ORDER) for f in inst]
    lab = np.zeros(NPIX, np.int32)
    for j, f in enumerate(rt, 1):
        lab[f] = j
    nlab = len(rt)
    gt = INST[image_id]
    gt_area = np.array([f.size for f in gt], np.float64)
    area = np.bincount(lab, minlength=nlab + 1).astype(np.float64)
    C = np.zeros((len(gt), nlab + 1), np.int64)
    for i, idx in enumerate(gt):
        C[i] = np.bincount(lab[idx], minlength=nlab + 1)
    dice = 2.0 * C / np.maximum(gt_area[:, None] + area[None, :], 1e-9)
    dice[:, 0] = 0.0
    gi, pi = np.nonzero(dice >= 0.5)
    o = np.argsort(-dice[gi, pi])
    ug, up, tp = set(), set(), 0
    for g, p in zip(gi[o], pi[o]):
        if g in ug or p in up:
            continue
        ug.add(g); up.add(p); tp += 1
    parity.append(2.0 * tp / (2.0 * tp + (nlab - tp) + (len(gt) - tp)))

direct9 = float(PER9[KEY9][:25].mean())
delta9 = abs(direct9 - float(np.mean(parity)))
print(f"parity | in-memory {direct9:.4f} | after RLE {np.mean(parity):.4f} | delta {delta9:.5f}")
print("!! encoding is losing score - fix this before submitting anything" if delta9 > 0.002
      else ">>> encoding is lossless")
del BL_VAL
gc.collect()

print("\nblending test maps ...", flush=True)
SSL_TEST = family_mean(SSL_MEMBERS, "test", TEST_IDS)
NON_TEST = family_mean(NON_MEMBERS, "test", TEST_IDS)
BL_TEST = blend(SSL_TEST, NON_TEST, BEST_W, TEST_IDS)
del SSL_TEST, NON_TEST
gc.collect()


def write_sub9(path, maps, ids, post, post_by_id=None):
    rows, counts, fallbacks = [], [], 0
    for image_id in ids:
        fg, seed = maps[image_id][0], maps[image_id][1]
        use = post if post_by_id is None else post_by_id.get(image_id, post)
        inst = instances_from_maps(fg, seed, use)
        if not inst:  # an image with GT and zero masks scores exactly 0.0
            fallbacks += 1
            inst = instances_from_maps(fg, seed, dict(t_cell=0.30, t_seed=0.30,
                                                      min_area=0, score_thr=0.0))[:1]
            if not inst:
                h = CFG.IMG_H // 2
                inst = [np.arange(h * CFG.IMG_W, h * CFG.IMG_W + 25)]
        counts.append(len(inst))
        rows.extend((image_id, flat_to_rle(f)) for f in inst)
    frame = pd.DataFrame(rows, columns=[ID_COL, ANN_COL])
    frame.insert(0, CFG.SUB_INDEX_COL, np.arange(len(frame), dtype=np.int64))
    frame.to_csv(path, index=False)
    assert set(frame[ID_COL].astype(str)) == set(ids), "Submission is missing test images."
    assert frame[ANN_COL].notna().all() and frame[CFG.SUB_INDEX_COL].is_unique
    return dict(path=str(path), rows=len(frame), mean=float(np.mean(counts)),
                min=int(np.min(counts)), max=int(np.max(counts)), fallbacks=fallbacks)


# Global-threshold blend -> THIS OVERWRITES the Stage 7 submission.csv.
info9 = write_sub9(WORK / "submission.csv", BL_TEST, TEST_IDS, POST9)
shutil.copy2(WORK / "submission.csv", ART / "submission_stage9_global.csv")
print("wrote (overwritten):", WORK / "submission.csv", "|", info9)

CANDS9 = [dict(name="stage9_blend_global", w=BEST_W, local_f1=GLOBAL9, **info9)]

if USE_DENS:
    dens_test = density_of(BL_TEST, TEST_IDS)
    bin_test = np.digitize(dens_test, EDGES)   # same edges as validation, unchanged
    post_by_id = {i: dens_post[int(b)] for i, b in zip(TEST_IDS, bin_test)}
    print("test bin sizes:", np.bincount(bin_test, minlength=3).tolist())
    info9d = write_sub9(ART / "submission_stage9_density.csv", BL_TEST, TEST_IDS,
                        POST9, post_by_id=post_by_id)
    CANDS9.append(dict(name="stage9_blend_density", w=BEST_W, local_f1=DENS9, **info9d))
    print("wrote:", info9d["path"], "|", info9d)

cand9 = pd.DataFrame(CANDS9).sort_values("local_f1", ascending=False)
cand9.to_csv(ART / "stage9_candidates.csv", index=False)
print("\n" + cand9.to_string(index=False))
print(f"\nGT mean instances/image {STATS['gt_mean_instances']:.1f} | "
      f"stage 9 predictions/image {info9['mean']:.1f}")
del BL_TEST
gc.collect()

parity | in-memory 0.8227 | after RLE 0.8227 | delta 0.00000
>>> encoding is lossless

blending test maps ...
wrote (overwritten): /kaggle/working/submission.csv | {'path': '/kaggle/working/submission.csv', 'rows': 20218, 'mean': 111.08791208791209, 'min': 11, 'max': 629, 'fallbacks': 0}

               name   w  local_f1                           path  rows       mean  min  max  fallbacks
stage9_blend_global 0.5  0.807216 /kaggle/working/submission.csv 20218 111.087912   11  629          0

GT mean instances/image 125.2 | stage 9 predictions/image 111.1


0

In [18]:
# ── S5 · Stage 9 report ───────────────────────────────────────────────────────
lines9 = [
    "# Stage 9 report - cross-family blend (SSL x non-SSL)",
    f"- SSL members (Stage 7): {SSL_MEMBERS} | non-SSL members (Stage 8): {NON_MEMBERS}",
    f"- validation identity verified against Stage 8 val_ids.json: {len(val_ids)} images, identical",
    f"- blend weight w = {BEST_W:.2f}  (w = non-SSL share; 0.0 = pure Stage 7, 1.0 = pure Stage 8)",
    "- blend-weight table:",
] + ["    " + l for l in wtab.to_string(index=False).splitlines()] + [
    f"- chosen post-processing: {POST9}",
    f"- local F1 (global thresholds): {GLOBAL9:.4f}",
    f"- local F1 (density-adaptive): {DENS9:.4f} -> {'used' if USE_DENS else 'rejected (+0.005 gate)'}",
    f"- RLE parity delta: {delta9:.5f}",
    f"- test predictions/image: mean {info9['mean']:.1f} "
    f"(min {info9['min']}, max {info9['max']}), fallbacks {info9['fallbacks']}",
    "",
    "## Prior points",
    "| stage | local | public |",
    "|---|---|---|",
    "| 5 (ImageNet ConvNeXt-UNet) | 0.8279 | 0.5293 |",
    "| 6 (SSL, starved schedule)  | 0.7740 | 0.5209 |",
    "| 7 (SSL x3 ensemble)        | 0.8035 | 0.5259 |",
    f"| 9 (blend, w={BEST_W:.2f})        | {GLOBAL9:.4f} | ? |",
    "",
    "Line through stages 2/4/5: public = 0.773 * local - 0.111. Stage 7 sat +0.016 above it,",
    "Stage 6 +0.033 - the SSL residual shrinks as local F1 rises, so do not credit this blend",
    "with a bonus it has not earned. Read it as sitting on or just above the line.",
    "",
    "## Submission order",
    "1. /kaggle/working/submission.csv - the blend at global thresholds.",
    "2. submission_stage9_density.csv only if the +0.005 gate accepted it.",
    "",
    "## Final selection",
    "Keep 5th_submission.csv (0.5293) as one of the two slots throughout - it lives on Kaggle's",
    "side and nothing lost locally affects it. The second slot goes to this blend even if it",
    "measures within ~0.005 of 0.5293: at 91 public images that gap is under one standard error,",
    "and a two-family blend has the better expected private score than either family alone.",
]
(ART / "STAGE9_REPORT.md").write_text("\n".join(lines9))
print("\n".join(lines9))
print("\nPrimary submission:", WORK / "submission.csv")
print("Total elapsed:", fmt(elapsed()))

# Stage 9 report - cross-family blend (SSL x non-SSL)
- SSL members (Stage 7): ['m1', 'm2', 'm3'] | non-SSL members (Stage 8): ['n1', 'n2']
- validation identity verified against Stage 8 val_ids.json: 86 images, identical
- blend weight w = 0.50  (w = non-SSL share; 0.0 = pure Stage 7, 1.0 = pure Stage 8)
- blend-weight table:
      w       f1  t_cell  t_seed  min_area  mean_pred
    0.5 0.814394    0.50    0.65        90 116.800000
    0.6 0.814305    0.50    0.70        90 116.911111
    0.7 0.814153    0.55    0.70        90 115.977778
    0.4 0.813390    0.50    0.65        90 116.733333
    0.3 0.812483    0.45    0.65        90 117.311111
    1.0 0.810416    0.55    0.70        90 116.622222
    0.0 0.810157    0.45    0.65        90 117.688889
- chosen post-processing: {'t_cell': 0.5, 't_seed': 0.7, 'min_area': 70, 'score_thr': 0.0}
- local F1 (global thresholds): 0.8072
- local F1 (density-adaptive): 0.8121 -> rejected (+0.005 gate)
- RLE parity delta: 0.00000
- test prediction